# IMPORTING AND CONFIGURING THE SERVER IN THE NOTEBOOK 

Here is where we make sure that everything is running smoothly and that the server is integrated into the notebook 

In [3]:
import json
import os
import platform
import re
import sys
import textwrap
import time
from contextlib import asynccontextmanager
from dataclasses import dataclass, field
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

REPO_DIR = Path.cwd()
load_dotenv(REPO_DIR / ".env")          # ANTHROPIC_API_KEY lives in .env, never in the notebook

# --- configuration -------------------------------------------------------------------------
SERVER_SCRIPT = next(iter(sorted(REPO_DIR.glob("mcp_server_week05_*.py"))), None)
assert SERVER_SCRIPT is not None, "Run this notebook from the folder containing mcp_server_week05_<lastname>.py"
LASTNAME = SERVER_SCRIPT.stem.removeprefix("mcp_server_week05_")

MODEL = os.getenv("ANTHROPIC_MODEL", "claude-sonnet-5-5")   # override in .env to pin another model
FIRST_OBJECT = "Betelgeuse"                                 # the object of my choice for the queries
SECOND_OBJECT = "Gj 667c"                          # the object of my choice for the queries
MAX_AGENT_TURNS = 8                                         # safety cap on LLM tool round researchings
DISPLAY_LIMIT = 2048                                        # max characters of one tool result printed
SERVER_LOG = REPO_DIR / "mcp_server_stderr.log"             # the server's stderr goes here

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found: copy .env.example to .env and add your key"
llm = anthropic.Anthropic()                                  # reads ANTHROPIC_API_KEY from the environment

print(f"{'Python':14s}{platform.python_version()}")
for package in ["mcp", "astroquery", "arxiv", "anthropic", "python-dotenv"]:
    print(f"{package:14s}{version(package)}")
print(f"{'LLM model':14s}{MODEL}")
print(f"{'Server script':14s}{SERVER_SCRIPT.name}")

Python        3.14.7
mcp           2.3.0
astroquery    0.4.11
arxiv         4.0.1
anthropic     1.15.0
python-dotenv 1.2.4
LLM model     claude-sonnet-5-5
Server script mcp_server_week05_alexis.py


# Starting the MCP Server and Properly Connecting to it for Application purposes 

In [4]:
def mcp_attr(obj, camel_name):
    """Read an MCP result fields regardless of capitalizations."""
    if hasattr(obj, camel_name):
        return getattr(obj, camel_name)
    snake_name = re.sub(r"([A-Z])", lambda m: "_" + m.group(1).lower(), camel_name)
    return getattr(obj, snake_name)

def server_parameters(env_overrides=None):
    """How to launch the MCP server: same Python interpreter, same folder, no LLM keys in its environment."""
    env = {k: v for k, v in os.environ.items() if not k.endswith("_API_KEY")}
    env.update(env_overrides or {})
    return StdioServerParameters(command=sys.executable, args=[str(SERVER_SCRIPT)], env=env, cwd=str(REPO_DIR))

@asynccontextmanager
# This function runs asynchronously which is required of the server 
async def mcp_session(env_overrides=None):
    """Start the server subprocess (stdio transport) and yield an initialised MCP ClientSession."""
    with open(SERVER_LOG, "a") as server_stderr:
        async with stdio_client(server_parameters(env_overrides), errlog=server_stderr) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                yield session

# Calling the server asking what tools it has nad printing out the tool descriptions for easy readibility 

In [5]:
async with mcp_session() as session:
    listing = await session.list_tools()
    for tool in listing.tools:
        print("=" * 100)
        print(f"TOOL: {tool.name}")
        print("=" * 100)
        print(tool.description)
        print("\nJSON Schema for the arguments:")
        print(json.dumps(mcp_attr(tool, "inputSchema"), indent=2))

TOOL: query_simbad
Look up one astronomical object in the SIMBAD database (CDS Strasbourg).

Returns a dict with:
  found                         true if SIMBAD matched the name, else false
  query                         the name that was looked up
  main_id                       SIMBAD's canonical identifier for the object
  object_type                   SIMBAD object-type code, e.g. "*" star, "G" galaxy,
                                "PN" planetary nebula, "Pu*" pulsating variable
  spectral_type                 MK spectral type string, e.g. "A0Va" (null for non-stars)
  parallax_mas                  trigonometric parallax in milliarcseconds
  parallax_error_mas            1-sigma uncertainty on the parallax
  radial_velocity_km_s          radial velocity in km/s (positive = receding)
  radial_velocity_error_km_s    1-sigma uncertainty on the radial velocity
  ra_deg, dec_deg               ICRS position in decimal degrees
  source                        data provenance

Any proper

# AI Agent Loop 

 Lets the AI Answer a question by calling my MCP tools that were generated using Claude AI. Claude decides which tool to call runs it to the server and repeats until a final answer is recorded. It returns the reason why the loop terminates whether because a final value was acquired or because it timed out/found no results

In [6]:
SYSTEM_PROMPT = ("You are an astronomy research assistant. You can call tools to query the SIMBAD astronomical "
    "database and to search arXiv. Use the tools when they help answer the question, and ground your "
    "answer in what they return.")
@dataclass
class ToolCall:
    turn: int
    tool: str
    arguments: dict
    result: object          # parsed result (dict / list) or the error text
    is_error: bool
    started_at: str
    seconds: float
@dataclass
class AgentRun:
    question: str
    tool_calls: list = field(default_factory=list)
    final_answer: str = ""
    turns: int = 0
    stop_reason: str = ""
def to_anthropic_tools(mcp_tools):
    """MCP tool declarations -> the `tools` format the Anthropic Messages API expects."""
    return [
        {"name": t.name, "description": t.description or "", "input_schema": mcp_attr(t, "inputSchema")}
        for t in mcp_tools
    ]
def unpack_tool_result(raw):
    """MCP CallToolResult -> (parsed data, text sent back to the LLM, is_error)."""
    is_error = bool(mcp_attr(raw, "isError"))
    texts = [c.text for c in raw.content if getattr(c, "type", "") == "text"]
    if is_error:
        message = " ".join(texts)
        return message, message, True
    structured = mcp_attr(raw, "structuredContent")
    if structured is not None:
        data = structured["result"] if set(structured) == {"result"} else structured
    else:
        parsed = [json.loads(t) for t in texts]
        data = parsed[0] if len(parsed) == 1 else parsed
    return data, json.dumps(data, ensure_ascii=False), False

def log_tool_call(call):
    """Print one tool call: name, input arguments and the returned result."""
    result = call.result
    # Don't print full paper text: show its size instead. Claude still receives all of it.
    if call.tool == "get_arxiv_fulltext" and isinstance(result, dict) and "text" in result:
        result = {**result, "text": f"[full text not shown: {len(result['text'])} characters sent to the LLM]"}

    body = result if isinstance(result, str) else json.dumps(result, indent=2, ensure_ascii=False)
    if len(body) > DISPLAY_LIMIT:
        body = body[:DISPLAY_LIMIT] + f"\n... [display truncated, {len(body) - DISPLAY_LIMIT} more characters]"
    status = "ERROR" if call.is_error else "ok"
    print(f"\n[turn {call.turn}] TOOL CALL  {call.tool}   ({status}, {call.seconds:.1f} s, {call.started_at} UTC)")
    print(f"  arguments: {json.dumps(call.arguments, ensure_ascii=False)}")
    print("  result:")
    print(textwrap.indent(body, "    "))

async def run_agent(session, question, system=SYSTEM_PROMPT, verbose=True):
    """Let the LLM call MCP tools until it gives a final answer; log every tool call along the way."""
    tools = to_anthropic_tools((await session.list_tools()).tools)
    messages = [{"role": "user", "content": question}]
    run = AgentRun(question=question)
    if verbose:
        print("=" * 100)
        print("QUESTION:", question)
        print("=" * 100)

    for turn in range(1, MAX_AGENT_TURNS + 1):
        run.turns = turn
        response = llm.messages.create(
            model=MODEL, max_tokens=2000, system=system, tools=tools, messages=messages
        )
        messages.append({"role": "assistant", "content": response.content})
        text = "".join(block.text for block in response.content if block.type == "text")
        tool_uses = [block for block in response.content if block.type == "tool_use"]
        run.stop_reason = response.stop_reason

        if not tool_uses:                         # plain-text answer: the loop is finished
            run.final_answer = text
            break

        if verbose and text.strip():
            print(f"\n[turn {turn}] LLM says: {text.strip()}")
        tool_results = []
        for use in tool_uses:
            started = datetime.now(timezone.utc)
            t0 = time.perf_counter()
            raw = await session.call_tool(use.name, dict(use.input))
            data, text_for_llm, is_error = unpack_tool_result(raw)
            call = ToolCall(turn, use.name, dict(use.input), data, is_error,
                            started.isoformat(timespec="seconds"), time.perf_counter() - t0)
            run.tool_calls.append(call)
            if verbose:
                log_tool_call(call)
            tool_results.append(
                {"type": "tool_result", "tool_use_id": use.id, "content": text_for_llm, "is_error": is_error}
            )
        messages.append({"role": "user", "content": tool_results})
    else:                                         # the for-loop never hit `break`
        run.stop_reason = "max_turns"
        run.final_answer = "(agent stopped: MAX_AGENT_TURNS reached without a final answer)"

    if verbose:
        print("\n" + "-" * 100)
        print(f"FINAL ANSWER  ({run.turns} LLM turns, {len(run.tool_calls)} tool calls)")
        print("-" * 100)
        print(run.final_answer)
    return run
RUNS = {}   # keeps each run so later sections can analyse it

# APPLICATION OF MCP WITH QUERY RESULTS 

In [7]:
COMPOUND_QUERY = (
    "What is the spectral type and parallax of {obj}, and what are the 3 most recent arXiv papers "
    "whose abstracts mention {obj}? Summarize what the papers suggest about its current evolutionary status."
)
async with mcp_session() as session:
    RUNS[FIRST_OBJECT] = await run_agent(session, COMPOUND_QUERY.format(obj=FIRST_OBJECT))

QUESTION: What is the spectral type and parallax of Betelgeuse, and what are the 3 most recent arXiv papers whose abstracts mention Betelgeuse? Summarize what the papers suggest about its current evolutionary status.

[turn 1] TOOL CALL  query_simbad   (ok, 2.0 s, 2026-10-20T17:19:47+00:00 UTC)
  arguments: {"object_name": "Betelgeuse"}
  result:
    {
      "found": true,
      "query": "Betelgeuse",
      "main_id": "* alf Ori",
      "object_type": "s*r",
      "spectral_type": "M1-M2Ia-Iab",
      "parallax_mas": 6.55,
      "parallax_error_mas": 0.8299999833106995,
      "radial_velocity_km_s": 21.91,
      "radial_velocity_error_km_s": 0.5099999904632568,
      "ra_deg": 88.79293899077537,
      "dec_deg": 7.407063995272694,
      "source": "SIMBAD (CDS) via astroquery.simbad TAP"
    }

[turn 1] TOOL CALL  search_arxiv   (ok, 0.3 s, 2026-10-20T17:19:49+00:00 UTC)
  arguments: {"keyword": "Betelgeuse", "max_results": 3}
  result:
    [
      {
        "title": "A Simple Miniaturi

In [8]:
COMPOUND_QUERY_Full = (
    "What is the spectral type and parallax of {obj}, and what are the 3 most recent arXiv papers "
    "whose abstracts mention {obj}? Summarize what the papers suggest about its current evolutionary status. Read the full texts and give me a full answer."
)
async with mcp_session() as session:
    RUNS[FIRST_OBJECT] = await run_agent(session, COMPOUND_QUERY_Full.format(obj=FIRST_OBJECT))

QUESTION: What is the spectral type and parallax of Betelgeuse, and what are the 3 most recent arXiv papers whose abstracts mention Betelgeuse? Summarize what the papers suggest about its current evolutionary status. Read the full texts and give me a full answer.

[turn 1] TOOL CALL  query_simbad   (ok, 1.7 s, 2026-10-20T17:20:28+00:00 UTC)
  arguments: {"object_name": "Betelgeuse"}
  result:
    {
      "found": true,
      "query": "Betelgeuse",
      "main_id": "* alf Ori",
      "object_type": "s*r",
      "spectral_type": "M1-M2Ia-Iab",
      "parallax_mas": 6.55,
      "parallax_error_mas": 0.8299999833106995,
      "radial_velocity_km_s": 21.91,
      "radial_velocity_error_km_s": 0.5099999904632568,
      "ra_deg": 88.79293899077537,
      "dec_deg": 7.407063995272694,
      "source": "SIMBAD (CDS) via astroquery.simbad TAP"
    }

[turn 1] TOOL CALL  search_arxiv   (ok, 0.2 s, 2026-10-20T17:20:30+00:00 UTC)
  arguments: {"keyword": "Betelgeuse", "max_results": 3}
  result:
   

# 4. Physical plausibility: is the SIMBAD output sensible? Test with 2nd Object

In [9]:
COMPOUND_QUERY_2 = (
    "What is the spectral type and parallax of {obj}, and what are the 3 most recent arXiv papers "
    "whose abstracts mention {obj}? Summarize what the papers suggest about its current evolutionary status. Read the full texts and give me a full answer."
)
async with mcp_session() as session:
    RUNS[SECOND_OBJECT] = await run_agent(session, COMPOUND_QUERY_2.format(obj=SECOND_OBJECT))

QUESTION: What is the spectral type and parallax of Gj 667c, and what are the 3 most recent arXiv papers whose abstracts mention Gj 667c? Summarize what the papers suggest about its current evolutionary status. Read the full texts and give me a full answer.

[turn 1] TOOL CALL  query_simbad   (ok, 1.6 s, 2026-10-20T17:21:00+00:00 UTC)
  arguments: {"object_name": "GJ 667C"}
  result:
    {
      "found": true,
      "query": "GJ 667C",
      "main_id": "HD 156384C",
      "object_type": "PM*",
      "spectral_type": "M1.5V",
      "parallax_mas": 138.0663,
      "parallax_error_mas": 0.028300000354647636,
      "radial_velocity_km_s": 6.368,
      "radial_velocity_error_km_s": 0.000699999975040555,
      "ra_deg": 259.7451137490841,
      "dec_deg": -34.99683685371723,
      "source": "SIMBAD (CDS) via astroquery.simbad TAP"
    }

[turn 1] TOOL CALL  search_arxiv   (ok, 0.2 s, 2026-10-20T17:21:02+00:00 UTC)
  arguments: {"keyword": "GJ 667C", "max_results": 10}
  result:
    [
      {

## COMPARISON WITH THE WEB

A pipeline can run flawlessly and still return nonsense (wrong object, wrong units, a stale value). So each SIMBAD result is tested against physical expectations **and** literature values. For GJ 667:

| Field | MCP server | SIMBAD website | Match? |
|---|---|---|---|
| Main ID | HD 156384C | HD 156384C | ✅ |
| Object type | PM* | High Proper Motion Star (PM*) | ✅ |
| Spectral type | M1.5V | M1.5V | ✅ |
| Parallax | 138.0663 ± 0.028300000354647636 mas | 138.0663 [0.0283] mas | ✅  |
| Radial velocity | 6.368 ± 0.000699999975040555 km/s | 6.368 [0.0007] km/s | ✅ |
| RA | 259.7451137490841° | 17h 18m 58.8272997802s | ✅ converts exactly to 259.74511374908° |
| Dec | −34.99683685371723° | −34° 59′ 48.612673382″ | ✅ converts exactly to −34.99683685° |

 SIMBAD WEBSITE : <https://simbad.harvard.edu/simbad/sim-id?Ident=GJ+667+C>, 


## 6. Is the LLM's summary consistent with the abstracts it was given?

An agent can retrieve the right documents and still misreport them. This section compares each final answer with the raw abstracts the arXiv tool returned. Three **mechanical** checks flag candidates for review:

1. **Relevance.** Does the object's name actually appear in each returned title/abstract? (The tool searches arXiv's `all` field, which includes comments and other metadata, so a hit does not guarantee the abstract mentions the object.)
2. **Unsupported numbers.** Numbers in the answer that appear in none of the abstracts, nor in the SIMBAD result, nor are derived from the SIMBAD parallax (distance = 1000/parallax).
3. **Phantom citations.** arXiv IDs in the answer that the tool never returned.

From reviewing the abstracts manually directly it is clear that the LLM Summary is consistent with the abstracts that were found.

# Deliberate failure modes
| ID | Layer attacked | Trigger |
|---|---|---|
| FM1 | max results request| maximum results requested is outside of mcp range |
| FM2 | arXiv keyword search | an ambiguous keyword  |


## FAILURE 1

In [37]:
async with mcp_session() as session:
    raw = await session.call_tool("search_arxiv", {"keyword": "Betelgeuse", "max_results": -300})
    _, direct_text, direct_is_error = unpack_tool_result(raw)

print("is_error:", direct_is_error)
print(direct_text)

is_error: True
Error executing tool search_arxiv: 1 validation error for search_arxivArguments
max_results
  Input should be greater than or equal to 1 [type=greater_than_equal, input_value=-300, input_type=int]
    For further information visit https://errors.pydantic.dev/2.14/v/greater_than_equal


### Error: Max results is outside of the range. I would implement a change saying that if a request for max results is outside the range that a random number within the accepted values would be chosen and the search would run from there. 

## FAILURE 2

In [52]:
Fail_OBJECT1 = "Sun"
COMPOUND_QUERY_Fail1 = " list all the stars with the same property as {obj}."

async with mcp_session() as session:
    RUNS[Fail_OBJECT1] = await run_agent(session, COMPOUND_QUERY_Fail1.format(obj=Fail_OBJECT1))


QUESTION:  list all the stars with the same property as Sun.

[turn 1] LLM says: "Same property as the Sun" is ambiguous, so I'll use the Sun's spectral type, G2V, as the property. SIMBAD can't return a full list of every G2V star, because it only looks up one named object at a time. Instead I'll check some well-known solar twins and analogs.

[turn 1] TOOL CALL  query_simbad   (ok, 1.8 s, 2026-10-20T18:40:09+00:00 UTC)
  arguments: {"object_name": "Sun"}
  result:
    {
      "found": false,
      "query": "Sun",
      "message": "SIMBAD has no object matching 'Sun'. Try another identifier (for example 'HD 172167', 'M 31', or a standard catalogue name)."
    }

[turn 1] TOOL CALL  query_simbad   (ok, 0.2 s, 2026-10-20T18:40:10+00:00 UTC)
  arguments: {"object_name": "18 Sco"}
  result:
    {
      "found": true,
      "query": "18 Sco",
      "main_id": "*  18 Sco",
      "object_type": "PM*",
      "spectral_type": "G2Va",
      "parallax_mas": 70.7371,
      "parallax_error_mas": 0.

In [58]:
Fail_OBJECT1 = "Betelzeus"
COMPOUND_QUERY_Fail1 = " Parallax of {obj}. return only the value"

async with mcp_session() as session:
    RUNS[Fail_OBJECT1] = await run_agent(session, COMPOUND_QUERY_Fail1.format(obj=Fail_OBJECT1))

QUESTION:  Parallax of Betelzeus. return only the value

[turn 1] TOOL CALL  query_simbad   (ok, 1.6 s, 2026-10-20T18:44:23+00:00 UTC)
  arguments: {"object_name": "Betelgeuse"}
  result:
    {
      "found": true,
      "query": "Betelgeuse",
      "main_id": "* alf Ori",
      "object_type": "s*r",
      "spectral_type": "M1-M2Ia-Iab",
      "parallax_mas": 6.55,
      "parallax_error_mas": 0.8299999833106995,
      "radial_velocity_km_s": 21.91,
      "radial_velocity_error_km_s": 0.5099999904632568,
      "ra_deg": 88.79293899077537,
      "dec_deg": 7.407063995272694,
      "source": "SIMBAD (CDS) via astroquery.simbad TAP"
    }

----------------------------------------------------------------------------------------------------
FINAL ANSWER  (2 LLM turns, 1 tool calls)
----------------------------------------------------------------------------------------------------
6.55 mas


##  Did not report that the values returned do not correspond to Betelzeus. Automatically applied an autocorrect where it may not be applicable. I would add a function that surveys if the key object name matches character for character not considering capitals and lowercases. 